# HuecoEnv: GRPO Training Loop (v3 — Hard Mode)

The LLM must output valid JSON trade offers or it **dies**.
No heuristic fallback. No free resources. Pure survival of the fittest.

In [ ]:
# Cell 1: Install dependencies
%pip uninstall -y vllm trackio
%pip install -U git+https://github.com/huggingface/trl.git git+https://github.com/meta-pytorch/OpenEnv.git bitsandbytes
%pip install --no-cache-dir --force-reinstall "vllm==0.17.1"

import importlib, traceback
try:
    vllm = importlib.import_module("vllm")
    print("vllm import OK:", vllm.__version__)
except Exception:
    print("vllm import failed.")
    traceback.print_exc()

print("If packages changed, restart the kernel before running the next cell.")

In [ ]:
# Cell 2: Login + environment flags
import os, importlib
from huggingface_hub import login

os.environ["TRL_EXPERIMENTAL_SILENCE"] = "1"
os.environ["VLLM_USE_V1"] = "0"

vllm = importlib.import_module("vllm")
print("Using vLLM:", vllm.__version__)

if os.getenv("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
else:
    login(add_to_git_credential=False)

In [ ]:
# Cell 3: Initialize the Environment
from train import HuecoGymWrapper

task_name = "adaptive_survival"
wrapper = HuecoGymWrapper(task_name=task_name, seed=42)
MAX_STEPS = 50
print(f"Environment ready: task={task_name}, max_steps={MAX_STEPS}")

In [ ]:
# Cell 4: Model + Tokenizer
from transformers import AutoTokenizer

model_name = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer loaded: {model_name}")

In [ ]:
# Cell 5: GRPO Configuration
from trl import GRPOConfig
from datasets import Dataset

dataset = Dataset.from_dict({"prompt": ["Solve the environment."] * 500})

prompt_len = 1400
completion_len = 128
use_vllm = True

cfg = {
    "num_train_epochs": 1,
    "learning_rate": 5e-6,
    "gradient_accumulation_steps": 8,
    "per_device_train_batch_size": 1,
    "warmup_steps": 20,
    "num_generations": 2,
    "max_completion_length": completion_len,
    "use_vllm": use_vllm,
    "output_dir": "huecoenv-grpo",
    "logging_steps": 1,
    "save_steps": 10,
}

if use_vllm:
    cfg["vllm_mode"] = "colocate"
    cfg["vllm_gpu_memory_utilization"] = 0.3

valid = set(getattr(GRPOConfig, "__dataclass_fields__", {}).keys())
if "max_prompt_length" in valid:
    cfg["max_prompt_length"] = prompt_len
elif "max_seq_length" in valid:
    cfg["max_seq_length"] = prompt_len + completion_len
elif "max_length" in valid:
    cfg["max_length"] = prompt_len + completion_len

cfg = {k: v for k, v in cfg.items() if k in valid}
grpo_config = GRPOConfig(**cfg)
print("GRPO config keys:", sorted(cfg.keys()))

In [ ]:
# Cell 6: HARD MODE Training Loop
from trl import GRPOTrainer
from trl.experimental.openenv import generate_rollout_completions
import json, re, csv, os

training_log = []

# ── Helper: Build a rich prompt from the current environment state ──
def build_state_prompt(obs_dict):
    return (
        "You are the Producer agent in a multi-agent resource economy. "
        "You must generate a JSON trade offer to acquire resources and survive.\n\n"
        f"Current State:\n"
        f"  Episode: {obs_dict.get('episode', 0)}, Step: {obs_dict.get('step', 0)}\n"
        f"  Your Compute: {obs_dict.get('producer_compute', 0):.1f}\n"
        f"  Your Data: {obs_dict.get('producer_data', 0):.1f}\n"
        f"  Your Trust Score: {obs_dict.get('producer_trust', 0.5):.2f}\n"
        f"  Survival Rate: {obs_dict.get('survival_rate', 0):.2f}\n\n"
        "Rules:\n"
        "  - You burn ~5 compute and ~4 data per step to produce artifacts.\n"
        "  - If BOTH your compute AND data drop below 1.0, you DIE.\n"
        "  - Trade wisely: request enough resources, but don't be too greedy.\n"
        "  - score_share: how much of your artifact score you share (0.0 - 1.0).\n"
        "    Lower values make the Allocator more likely to accept your trade.\n\n"
        "Respond with ONLY a valid JSON object in this exact format:\n"
        '{"compute": <float>, "data": <float>, "want": {"score_share": <float>}}\n\n'
        "Your trade offer:"
    )


# ── Helper: Parse the LLM's text output into a trade action dict ──
def parse_trade_json(text):
    try:
        match = re.search(r'\{[^{}]*"compute"[^{}]*\}', text)
        if not match:
            match = re.search(r'\{[^{}]*\}', text)
        if match:
            parsed = json.loads(match.group())
            compute = max(0.0, min(50.0, float(parsed.get("compute", 5.0))))
            data = max(0.0, min(40.0, float(parsed.get("data", 3.0))))
            want = parsed.get("want", {})
            if isinstance(want, dict):
                score_share = max(0.0, min(1.0, float(want.get("score_share", 0.3))))
            else:
                score_share = 0.3
            return {
                "compute": compute,
                "data": data,
                "want": {"score_share": score_share},
            }
    except (json.JSONDecodeError, ValueError, TypeError, AttributeError):
        pass
    return None


# ── The POISON offer: guarantees trade rejection and death ──
POISON_OFFER = {"compute": 0.0, "data": 0.0, "want": {"score_share": 0.99}}


# ── The REAL rollout function (HARD MODE) ──
def rollout_func(prompts, trainer=None):
    episode_prompt_ids = []
    episode_completion_ids = []
    episode_logprobs = []
    correctness_rewards = []

    for prompt_text in prompts:
        obs = wrapper.reset()
        total_reward = 0.0
        steps_survived = 0
        done = False

        # Build a state-aware prompt
        state_prompt = build_state_prompt(obs)

        # Generate a trade offer from the LLM
        rollout_outputs = generate_rollout_completions(trainer, [state_prompt])[0]
        action_text = rollout_outputs.get("text", "")

        # Parse the LLM's JSON output
        parsed_action = parse_trade_json(action_text)

        if parsed_action is not None:
            total_reward += 0.5  # Reward for valid JSON
        else:
            total_reward -= 2.0  # Penalty for bad JSON

        # ── Run the FULL 50-step episode ──
        while not done and steps_survived < MAX_STEPS:
            # KEY CHANGE: If JSON parsing failed, pass a POISON offer
            # that requests nothing and demands 99% share.
            # The Allocator will ALWAYS reject this, so the producer
            # gets ZERO resources and slowly starves to death.
            # NO HEURISTIC FALLBACK. The LLM must learn or die.
            if parsed_action is not None:
                action_overrides = {"producer_0": parsed_action}
            else:
                action_overrides = {"producer_0": POISON_OFFER}

            obs, reward, done, info = wrapper.step(action_overrides)
            total_reward += reward
            steps_survived += 1

        # Survival bonus/penalty
        if steps_survived >= MAX_STEPS:
            total_reward += 5.0
        else:
            total_reward -= 3.0

        # Record metrics
        brain_info = wrapper.end_episode()
        survival_rate = brain_info.get("survival_rate", 0.0)
        training_log.append({
            "survival_rate": survival_rate,
            "steps_survived": steps_survived,
            "json_ok": 1 if parsed_action is not None else 0,
        })

        episode_prompt_ids.append(rollout_outputs["prompt_ids"])
        episode_completion_ids.append(rollout_outputs["completion_ids"])
        episode_logprobs.append(rollout_outputs["logprobs"])
        correctness_rewards.append(total_reward)

    return {
        "prompt_ids": episode_prompt_ids,
        "completion_ids": episode_completion_ids,
        "logprobs": episode_logprobs,
        "correct_reward": correctness_rewards,
    }


def reward_correct(completions, **kwargs):
    rewards = kwargs.get("correct_reward")
    return [float(r) for r in rewards] if rewards else [0.0] * len(completions)


# ── Build the Trainer ──
trainer = GRPOTrainer(
    model=model_name,
    processing_class=tokenizer,
    reward_funcs=[reward_correct],
    train_dataset=dataset,
    args=grpo_config,
    rollout_func=rollout_func,
)

print("Starting TRL GRPO Training (HARD MODE)...")
trainer.train()

# ── Save results ──
os.makedirs("data", exist_ok=True)
with open("data/training_adaptive_survival.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["episode", "survival_rate"])
    for idx, row in enumerate(training_log):
        writer.writerow([idx + 1, row["survival_rate"]])

print(f"Training Complete! {len(training_log)} episodes logged.")
if training_log:
    print(f"Final survival rate: {training_log[-1]['survival_rate']:.2%}")

In [ ]:
!pip install matplotlib

In [ ]:
# Cell 7: Plot the results
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/training_adaptive_survival.csv")

plt.figure(figsize=(10, 6))
plt.style.use('dark_background')

if df['survival_rate'].nunique() > 1:
    df['smooth'] = df['survival_rate'].rolling(window=30, min_periods=1).mean()
    plt.plot(df['episode'], df['survival_rate'], color='#00ffcc', alpha=0.25, lw=1)
    plt.plot(df['episode'], df['smooth'], color='#00ffcc', lw=2.5, label='Rolling Avg')
else:
    plt.plot(df['episode'], df['survival_rate'], color='#00ffcc', lw=2.5, label='Survival Rate')

plt.title('Qwen3-1.7B GRPO Training: Agent Survival Rate', fontsize=16, pad=15)
plt.xlabel('Training Episode', fontsize=12)
plt.ylabel('Survival Rate', fontsize=12)
plt.ylim(0.0, 1.05)
plt.legend()
plt.grid(alpha=0.15)
plt.tight_layout()

os.makedirs('assets', exist_ok=True)
plt.savefig('assets/survival_plot.png', dpi=300)
plt.show()
print('Graph saved to assets/survival_plot.png')